# Error analysis

Which queries LLM2Vec wins or loses, against the two spring baselines we show in the demo: BM25 and bge-small.
Like `reproduce.ipynb`, it only reads the files in `results/` (and the testbed data for the texts).
`full` is our testbed row (Qwen2.5-0.5B, full recipe), `q15-full` the same recipe on Qwen2.5-1.5B, `sllama` the authors' adapters.

In [1]:
import numpy as np
import pandas as pd

import analysis
from analysis import testbed

P = analysis.per_query()
pd.set_option("display.max_colwidth", 90)


def ndcg(system, name):
    return np.array(P[name][system]["ndcg10"])


def rank(system, name):
    return np.array(P[name][system]["rank"])  # 101 means: not in the top 100

## 1. Wins and losses per query

A query is a win (or a loss) when its nDCG@10 changes by more than 0.01.

In [2]:
rows = []
for name in testbed.DATASETS:
    for s in ["full", "q15-full", "sllama"]:
        for b in ["bm25", "bge"]:
            d = ndcg(s, name) - ndcg(b, name)
            rows.append({"dataset": name, "ours": s, "against": b, "wins": int((d > 0.01).sum()),
                         "ties": int((abs(d) <= 0.01).sum()), "losses": int((d < -0.01).sum())})
pd.DataFrame(rows)

,dataset,ours,against,wins,ties,losses
0,scifact,full,bm25,48,146,106
1,scifact,full,bge,33,154,113
2,scifact,q15-full,bm25,63,139,98
3,scifact,q15-full,bge,47,139,114
4,scifact,sllama,bm25,33,102,165
5,scifact,sllama,bge,27,100,173
6,nfcorpus,full,bm25,44,102,177
7,nfcorpus,full,bge,32,98,193
8,nfcorpus,q15-full,bm25,64,99,160
9,nfcorpus,q15-full,bge,53,92,178


## 2. Word overlap: where do we lose?

For each query we take the part of its word stems (BM25 preprocessing of the testbed) that we find in its best labelled document. Then we cut the queries in three groups of the same size, from low to high overlap.

In [3]:
rows = []
for name in testbed.DATASETS:
    ov = analysis.overlap(name)
    cuts = np.quantile(ov, [1 / 3, 2 / 3])
    group = np.digitize(ov, cuts)
    for s in ["bm25", "bge", "full", "q15-full", "sllama"]:
        row = {"dataset": name, "system": s}
        for k, label in enumerate(["low overlap", "middle", "high overlap"]):
            row[label] = ndcg(s, name)[group == k].mean()
        rows.append(row)
    print(f"{name}: limits of the groups {np.round(cuts, 2)}")
overlap_table = pd.DataFrame(rows).set_index(["dataset", "system"])
overlap_table

scifact: limits of the groups [0.43 0.69]


nfcorpus: limits of the groups [0.5 1. ]


arguana: limits of the groups [0.15 0.21]


low overlap    middle  high overlap
dataset  system                                       
scifact  bm25         0.320558  0.743611      0.960137
         bge          0.429949  0.766966      0.929335
         full         0.262609  0.588711      0.804331
         q15-full     0.357179  0.685635      0.728633
         sllama       0.157474  0.399720      0.576455
nfcorpus bm25         0.047831  0.288422      0.535167
         bge          0.093874  0.322852      0.525622
         full         0.041653  0.170019      0.303789
         q15-full     0.064202  0.222381      0.322772
         sllama       0.045051  0.148310      0.249486
arguana  bm25         0.305896  0.515878      0.724469
         bge          0.467054  0.631101      0.718982
         full         0.269807  0.459740      0.626499
         q15-full     0.357143  0.530578      0.655106
         sllama       0.255034  0.444135      0.628236

In [4]:
# the gap to BM25 in each group
gap = overlap_table.copy()
for name in testbed.DATASETS:
    gap.loc[name] = (overlap_table.loc[name] - overlap_table.loc[(name, "bm25")]).values
gap.drop(index="bm25", level="system").style.format("{:+.3f}")

This is the clearest result of the analysis. **Our models lose most of their points on the queries that share many words with the right document**, where BM25 is very strong.
On the queries with few shared words, LLM2Vec is close to BM25, and the 1.5B model is even better than BM25 (SciFact +0.04, ArguAna +0.05).
bge-small is close to BM25 or better in all groups: it learned exact matching from its supervised training, our unsupervised SimCSE did not.

## 3. Hubs: does every query get the same documents?

Number of different documents in all the top 10 lists together (the maximum is 10 × number of queries). A small number means that a few documents come back for every query.

In [5]:
rows = []
for name in ["scifact", "nfcorpus"]:
    n = 10 * len(P[name]["qids"])
    for s in ["uni", "bi", "bi-mntp", "uni-simcse", "bi-simcse", "full", "q15-full", "sllama",
              "ettin", "ettin-simcse", "ettin-simcse-d01", "modernbert", "modernbert-simcse", "modernbert-simcse-d01"]:
        rows.append({"dataset": name, "system": s, "distinct docs in top 10": P[name][s]["distinct_top10"], "of": n})
pd.DataFrame(rows).pivot(index="system", columns="dataset", values="distinct docs in top 10")

dataset,nfcorpus,scifact
system,,
bi,463,1484
bi-mntp,634,1747
bi-simcse,1386,1892
ettin,208,803
ettin-simcse,88,399
ettin-simcse-d01,22,39
full,1392,1939
modernbert,142,746
modernbert-simcse,206,1010


The raw decoder in causal mode (`uni`) gives the same documents to many queries, each step of LLM2Vec makes the lists more different.
The encoders after our SimCSE are the extreme case: with dropout 0.1, Ettin returns only 39 different documents for the 300 SciFact queries (see `reproduce.ipynb`, Section 3).

## 4. Concrete queries

The SciFact queries where `full` gains or loses the most against bge, with the rank of the first labelled document for each system.

In [6]:
corpus, queries, qrels = testbed.load("scifact")
qids = P["scifact"]["qids"]
ov = analysis.overlap("scifact")


def examples(order, n=8):
    rows = []
    for i in order[:n]:
        q = qids[i]
        doc = next(d for d, g in qrels[q].items() if g > 0)
        rows.append({"id": q, "claim": queries[q], "overlap": round(ov[i], 2),
                     **{s: rank(s, "scifact")[i] for s in ["bm25", "bge", "full", "q15-full"]},
                     "labelled document": corpus[doc]["title"][:80]})
    return pd.DataFrame(rows).set_index("id")


d = ndcg("full", "scifact") - ndcg("bge", "scifact")
print("biggest wins against bge")
examples(np.argsort(-d))

biggest wins against bge


,claim,overlap,bm25,bge,full,q15-full,labelled document
id,,,,,,,
1100,Statins increase blood cholesterol.,0.75,2,13,1,1,A Century of Cholesterol and Coronaries: From Plaques to Genes to Statins
1024,Recurrent mutations occur frequently within CTCF anchor sites adjacent to oncogenes.,0.70,1,7,1,101,3D Chromosome Regulatory Landscape of Human Pluripotent Cells.
1099,Statins decrease blood cholesterol.,0.75,3,6,1,1,A Century of Cholesterol and Coronaries: From Plaques to Genes to Statins
1,0-dimensional biomaterials show inductive properties.,0.00,101,13,2,2,New opportunities: the use of nanotechnologies to manipulate and track stem cell
1272,The single flash-evoked ERG b-wave is generated by activity of ON-bipolar cells.,0.33,9,20,2,2,Specific deficit of the ON response in visual transmission by targeted disruptio
507,Helminths interfere with immune system control of macrophages activated by IL-4 favor ...,0.58,2,5,1,2,T helper 2 cytokines inhibit autophagic control of intracellular Mycobacterium t
845,Neutrophil extracellular traps (NETs) are released by ANCA-stimulated neutrophils.,1.00,1,4,1,1,Netting neutrophils in autoimmune small-vessel vasculitis
212,CR is associated with higher methylation age.,0.60,9,4,1,4,Caloric restriction delays age-related methylation drift


In [7]:
print("biggest losses against bge")
examples(np.argsort(d))

biggest losses against bge


,claim,overlap,bm25,bge,full,q15-full,labelled document
id,,,,,,,
839,Nanoparticles can be targeted against specific cell types by incorporating aptamers in...,0.75,1,1,41,3,Aptamer-functionalized lipid nanoparticles targeting osteoblasts as a novel RNA
1279,The treatment of cancer patients with co-IR blockade precipitates adverse autoimmune e...,0.40,8,1,39,2,Is autoimmunity the Achilles' heel of cancer immunotherapy?
1014,Rapamycin decreases the concentration of triacylglycerols in fruit flies.,0.50,1,1,101,10,Mechanisms of Life Span Extension by Rapamycin in the Fruit Fly Drosophila melan
993,Pyridostatin destabilizes the G - quadruplex in the telomeric region.,0.60,1,1,12,5,Targeting BRCA1 and BRCA2 Deficiencies with G-Quadruplex-Interacting Compounds
960,Polymeal nutrition reduces cardiovascular mortality.,0.60,1,1,67,13,"The Polymeal: a more natural, safer, and probably tastier (than the Polypill) st"
268,Cold exposure increases BAT recruitment.,0.80,2,1,24,36,Cold Exposure Promotes Atherosclerotic Plaque Growth and Instability via UCP1-De
72,Activator-inhibitor pairs are provided dorsally by Admpchordin.,0.50,1,1,37,1,Regulation of ADMP and BMP2/4/7 at Opposite Embryonic Poles Generates a Self-Reg
1278,The treatment of cancer patients with co-IR blockade does not cause any adverse autoim...,0.40,10,1,35,2,Is autoimmunity the Achilles' heel of cancer immunotherapy?


Most losses look the same: a claim with a rare word (aptamers, rapamycin, pyridostatin, Polymeal, Admp) that is written in the labelled abstract. BM25 and bge put this abstract first, we put it somewhere from rank 12 to out of the top 100.
Our model finds papers on the right topic, but not the one with the exact words. The 1.5B model puts most of them much higher (839: from 41 to 3, 1279: from 39 to 2, 72: from 37 to 1).
The wins are more mixed. Some claims share many words with the abstract but bge puts it lower (the statins). Others share almost no word, like the hero query of the demo below (query 1) or query 1272.
Queries 1099 and 1100 say the opposite thing (statins *increase* or *decrease* cholesterol) and we find the same abstract at rank 1 for both: SciFact labels a document for a claim when it supports **or** refutes it, so a model blind to the negation is not punished here.

## 5. The demo queries

We take them from this analysis (all on SciFact, `demo.py`):

* **Hero**, query 1: *0-dimensional biomaterials show inductive properties.* The labelled abstract (*New opportunities: the use of nanotechnologies to manipulate and track stem cells*) shares no word stem with the claim. BM25 can not find it, bge puts it at rank 13, LLM2Vec at rank 2.
* **Failure**, query 644: *Insulin increases risk of severe kidney failure.* All the words of the claim are in the labelled abstract (*Diabetes treatments and risk of amputation, blindness, severe kidney failure...*). BM25 and bge put it first, LLM2Vec at rank 79 (77 in the live demo, the claim is encoded alone and bfloat16 changes the last digits): it returns kidney disease papers. The abstract is about many diabetes treatments and outcomes, and one mean vector keeps this main topic, not one item of the list.
* **Wildcard** backups: query 623 on vitamin D and multiple sclerosis, then our rewording without the key words (*People who get little sunshine are more likely to develop MS.*), then query 1272 where we beat both baselines.

In [8]:
for q in ["1", "644", "623", "1272"]:
    i = qids.index(q)
    print(q, {s: int(rank(s, "scifact")[i]) for s in ["bm25", "bge", "full", "q15-full", "sllama"]}, "overlap", round(ov[i], 2))

1 {'bm25': 101, 'bge': 13, 'full': 2, 'q15-full': 2, 'sllama': 3} overlap 0.0
644 {'bm25': 1, 'bge': 1, 'full': 79, 'q15-full': 7, 'sllama': 75} overlap 1.0
623 {'bm25': 11, 'bge': 3, 'full': 3, 'q15-full': 23, 'sllama': 14} overlap 0.33
1272 {'bm25': 9, 'bge': 20, 'full': 2, 'q15-full': 2, 'sllama': 29} overlap 0.33
